In [ ]:
import numpy as np
import bacco

In [ ]:
import os
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

snap = 264
zoom = {}

loaded = []
for i in range(len(name_list)):
    if i<30:
        base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/LH_{:d}/hydro_output/".format(i)
    else:
        base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), use_ids=True, numpart=4320)


In [ ]:
for i, key in enumerate(zoom['fiducial'].sub.keys()): print(i, key)

In [ ]:
# Load the Halo Selection
with open("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []

    for line in f.readlines():
        final_sel.append(int(line.split()[0]))

final_sel = np.array(final_sel)

# Perform the cross-match with MTNG halos
xmatch = {}

for i in range(len(name_list)):
    xmatch[name_list[i]] = utils.cross_match(zoom[name_list[i]], snap=264, name=name_list[i])

# Load MTNG and get the fraction of halos to do the upweighting
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)

m200b = np.log10(1e10 * mtng.fof['halo_m200b'])

mbins = np.concatenate(
    (np.arange(11, 11.5, 0.0025),
    np.arange(11.5, 12.5, 0.005),
    np.arange(12.5, 13.5, 0.025),
    np.arange(13.5, 15.01, 0.125))
)

h_frac = np.zeros(len(final_sel))
for m in range(len(mbins)-1):
    h_frac[m] = np.where(( m200b[final_sel]>=mbins[m]) & ( m200b[final_sel]<mbins[m+1]))[0].shape[0] / \
             np.where(( m200b>=mbins[m]) & ( m200b<mbins[m+1]))[0].shape[0]

zoom_split = {}
zoom_sel = {}
for i in range(len(name_list)):
    zoom_split[name_list[i]] = utils.split_halos(zoom[name_list[i]])

    zoom_sel[name_list[i]] = {}

    zoom_sel[name_list[i]]['sel'] = xmatch[name_list[i]]['ind'][:,np.newaxis,np.newaxis]
    zoom_sel[name_list[i]]['h_frac'] = h_frac[np.newaxis, :]

In [ ]:
zoom_sel['fiducial']['sel'].shape

In [ ]:
zoom_split['fiducial'].rhalf_m2half(sel_mask=zoom_sel['fiducial'], nbins=20)

In [ ]:
def SFR_mstar(self, sel_mask=None, nbins=100):

    # I don't really want this as a function of mass, but instead of redshift I believe
    bins = np.logspace(8, 13, nbins)

    first = self.sim.fof['halo_firstsub']
    nsubs = self.sim.fof['halo_nsubs']
    
    counts     = np.zeros(nbins-1)
    weights    = np.zeros(nbins-1)
    sSFR_mean   = np.zeros(nbins-1)
    mstar_mean = np.zeros(nbins-1)

    for m in range(len(sel_mask['sel'])):
        if sel_mask['h_frac'][0][m]!=0:

            mstar = []
            sSFR  = []
            for i in range(len(sel_mask['sel'][m][0])):
                # Get all the subhalos inside of a certain halo in the selection
                mstar.extend(self.sim.sub['MassType'][:,4][first[sel_mask['sel'][m][0][i]]:first[sel_mask['sel'][m][0][i]]+nsubs[sel_mask['sel'][m][0][i]]])
                sSFR.extend(self.sim.sub['SFR'][first[sel_mask['sel'][m][0][i]]:first[sel_mask['sel'][m][0][i]]+nsubs[sel_mask['sel'][m][0][i]]])

            mstar = 1e10 * np.array(mstar)
            sSFR  = 1e10 * np.array(sSFR) / mstar

            ids = np.digitize(mstar, bins)

            counts_i = np.array([np.sum( np.ones(len(mstar))[np.where(ids==j)]) for j in range(1,len(bins))])
            weights += counts_i / sel_mask['h_frac'][0][m]
            counts  += counts_i

            sSFR_mean += np.array([np.sum(sSFR[np.where(ids==j)]) for j in range(1,len(bins))]) / sel_mask['h_frac'][0][m]
            mstar_mean += np.array([np.sum(mstar[np.where(ids==j)]) for j in range(1,len(bins))])

    sSFR_mean /= weights
    mstar_mean /= counts

    return {'sSFR':sSFR_mean, 'mstar':mstar_mean, 'counts':counts}

In [ ]:
mhalo_edges = np.logspace(11, 14.5, )